In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import load_workbook
import matplotlib
matplotlib.use('cairo')  # Use a non-interactive backend for saving plots
# Load and reshape data
def load_and_melt_data(file_path, media_label):
    wb = load_workbook(filename=file_path, data_only=True)
    full_df = pd.DataFrame()

    for sheet_name in wb.sheetnames:
        sheet = pd.read_excel(file_path, sheet_name=sheet_name)
        essential_counts = sheet.notna().sum()
        model_df = pd.DataFrame({
            'Category': essential_counts.index,
            'Essential_Genes': essential_counts.values,
            'Model': sheet_name,
            'Media': media_label
        })
        full_df = pd.concat([full_df, model_df], ignore_index=True)

    full_df = full_df.dropna(subset=['Essential_Genes'])
    return full_df

# Plot grouped bar chart with unified y-limit
def plot_grouped_bar(data, media_label, unified_max, save_path_prefix):
    data_media = data[data['Media'] == media_label]

    plt.figure(figsize=(22, 12))
    sns.barplot(data=data_media, x='Model', y='Essential_Genes', hue='Category')
    plt.title(f'Essential Genes per Category for Each Model ({media_label} Media)', fontsize=22)
    plt.xticks(rotation=45, ha='right', fontsize=14)
    plt.yticks(fontsize=14)
    plt.xlabel('Category', fontsize=18)
    plt.ylabel('Number of Essential Genes', fontsize=18)
    plt.ylim(0, unified_max + 5)  # +5 for visual breathing room
    plt.legend(title='Model', title_fontsize=16, fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{save_path_prefix}_{media_label}.pdf")
    plt.show()

# File paths
file_path_rich = './EssentialityDataCategorizedAllPTYG.xlsx'
file_path_minimal = './EssentialityDataCategorizedAllSuccinate.xlsx'

# Load and combine data
df_rich = load_and_melt_data(file_path_rich, 'Rich')
df_minimal = load_and_melt_data(file_path_minimal, 'Minimal')
combined_df = pd.concat([df_rich, df_minimal], ignore_index=True)

# Compute max across both media
unified_max = combined_df['Essential_Genes'].max()

# Use desired path prefix for saving
plot_grouped_bar(combined_df, 'Rich', unified_max, save_path_prefix='essential_genes_per_category')
plot_grouped_bar(combined_df, 'Minimal', unified_max, save_path_prefix='essential_genes_per_category')

